In [2]:
import pandas as pd
import re
import pickle
import os
import unicodedata
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
data = pd.read_csv("../data/np20ng.csv")

In [4]:
data.head()

,source,category,heading,content
0,Gorkhapatra,Education,"अटेर गर्दै कतिपय विद्यालय, चैतमै भर्ना अभियान ...","काठमाडौं,चैत्र ७ गते । उपत्याकाका कतिपय विद्या..."
1,Gorkhapatra,Education,"निर्वाचन प्रचारमा विद्यालय, शिक्षक र विद्यार्...","प्रकाश सिलवाल काठमाडौँ, चैत ६ गते । विगतका निर..."
2,Gorkhapatra,Education,विद्यार्थीलाई परम्परागत सीप सिकाउँदै विद्यालय,"अमरराज नहर्की तनहुँ, चैत्र ५ गते । तनहुँको एक ..."
3,Gorkhapatra,Education,आन्दोलन नगर्न शिक्षामन्त्रीको आग्रह,"गोरखापत्र समाचारदाता काठमाडौँ, चैत ३ गते । शिक..."
4,Gorkhapatra,Education,विश्वविद्यालयमा हडताल नगर्न शिक्षामन्त्रीको आग्रह,"काठमाडौं, चैत २ गते। शिक्षा विज्ञान तथा प्रविध..."


In [5]:
def validate_dataframe(df):
    initial_len = len(df)

    # drop rows where the main content is empty or whitespace-only
    df = df.dropna(subset=["content"])
    df = df[df["content"].str.strip().ne("")]

    # drop exact duplicates on (heading + content)
    df = df.drop_duplicates(subset=["heading", "content"])

    # flag rows where content is mostly non-Devanagari
    def devanagari_ratio(text: str) -> float:
        devs = sum(1 for ch in text if "\u0900" <= ch <= "\u097f")
        return devs / max(len(text), 1)

    df["dev_ratio"] = df["content"].apply(devanagari_ratio)
    low_quality = df[df["dev_ratio"] < 0.3]

    if len(low_quality) > 0:
        print(
            f"{len(low_quality)} rows have <30% Devanagari characters — possible encoding issues."
        )
        print(low_quality[["source", "heading", "dev_ratio"]].head(10))

    df = df.drop(columns=["dev_ratio"])

    print(
        f"Validation: {initial_len} → {len(df)} rows ({initial_len - len(df)} removed)"
    )
    return df

data = validate_dataframe(data)

18 rows have <30% Devanagari characters — possible encoding issues.
              source                                            heading  \
8500       Ekantipur  अमेरिकामा भारतीय-अमेरिकन इन्जिनियरमाथि गोली प्...   
86163   Onlinekhabar  विश्वकपको खेल तालिका सार्वजनिक : भारत र पाकिस्...   
111620   Nepalkhabar  व्यापारिकरुपमा बनाइएको यान पहिलोपटक अन्तरिक्ष ...   
118117      Setopati  ट्विटरमा आन्तरिक समस्या पछि पासवर्ड परिवर्तन ग...   
196592      Setopati  हुन्डाइ गाडीको मूल्य बढ्यो, कुन मोडलको कति पर्...   
196603      Setopati  साउन १६ सम्म यो मूल्यमा पाइन्छ हुन्डाइ गाडीहरू...   
197993  Onlinekhabar  टाटाले जडान गर्‍यो ११६ स्थानमा विद्युतीय गाडी ...   
200680      Setopati           विवाह बन्धनमा बाँधिए काजल अग्रवाल र गौतम   
200689      Setopati              नेहा कक्करले गरिन् रोहनप्रितसँग विवाह   
203168      Setopati  'रकी'ले अभिनेता भएको वास्तविक अनुभूति दिलायो :...   

        dev_ratio  
8500     0.000000  
86163    0.114097  
111620   0.287105  
118117   0.000000  
196592

In [6]:
def clean_and_format(entry: dict) -> str:

    # combine header and body
    header = f"Category: {entry.get('category', 'General')} | Title: {entry.get('heading', '')}"
    body = entry.get("content", "")
    full_text = f"{header}\n{body}"

    # unicode normalization
    full_text = unicodedata.normalize("NFKC", full_text)

    # fix known encoding artifact
    full_text = full_text.replace("¥", "र्")

    # collapse whitespace
    full_text = re.sub(r"\s+", " ", full_text).strip()

    return full_text

In [7]:
# process and chunk
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ।", "।", "|", " ", ""],
    chunk_size=1000,
    chunk_overlap=200,
)

In [8]:
all_chunks = []
for row in data.itertuples(index=False):
    entry = row._asdict()
    cleaned_text = clean_and_format(entry)

    metadata = {
        "source": entry.get("source", "Unknown"),  # outlet name
        "category": entry.get("category", "General"),  # news category
        "heading": entry.get("heading", ""),  # article title
    }

    # create chunks
    doc_chunks = text_splitter.create_documents([cleaned_text], metadatas=[metadata])
    all_chunks.extend(doc_chunks)

print(f"Total Source Docs: {len(data)}")
print(f"Total Chunks Generated: {len(all_chunks)}")
print(f"Average Chunks per Doc: {len(all_chunks)/len(data):.2f}")

Total Source Docs: 231128
Total Chunks Generated: 593779
Average Chunks per Doc: 2.57


In [9]:
# save chunks
save_path = "../data/chunks/nepali_news_chunks.pkl"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "wb") as f:
    pickle.dump(all_chunks, f)